# Assignment 2: Language Modeling

## Overview

Instruction-tuning is a supervised fine-tuning technique that trains language models on a dataset of <instruction, output> pairs to improve their ability to follow user prompts. The goal of this assignment is to perform instruction-tuning for a less capable model, i.e., Qwen1.5-0.5B, using a instruction dataset synthetically generated by a more capable model (GPT-5.2).

Let a training example consist of a prompt $x$ (instruction) and a response $y$, tokenized into:

$$x = (x_1, \ldots, x_m), \quad y = (y_1, \ldots, y_n)$$

Concatenated into the full input sequence:

$$s = (s_1, \ldots, s_T) = (x_1, \ldots, x_m, y_1, \ldots, y_n), \quad T = m + n$$

An auto-regressive language model with parameters $\theta$ computes the token-level cross-entropy loss:

$$\mathcal{L}_\theta = -\log P(s_t \mid s_{<t})$$

The dataset contains **1,097 synthetic `<instruction, output>` pairs**, split **70 / 15 / 15** into train, dev, and test sets.

## Requirements

- Implement each task in the cells marked **TODO: your code here**
- You must implement the training and evaluation using plain PyTorch only. Frameworks such as HuggingFace `transformers`, `trl`, or others are **not allowed**.
- The tokenizer and model may be loaded via HuggingFace `transformers`\ `peft`.
- Save your instruction-tuned models as **you will need them for Assignment 3**.

## Submission
- Submit only the .ipynb file. No need to submit any models.

## Deadline
**April 2, 23:59 CET**

## Note: GPU Resources

**Test your code on CPU first.** Work on CPU first to make sure everything runs before switching to GPU for training. GPU time is limited — do not waste it debugging code that could be verified on CPU.



## Install dependencies

In [1]:
!pip install transformers datasets torch accelerate huggingface_hub tqdm peft

## Imports & device setup

In [2]:
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, DatasetDict
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home/spark-dasha/miniconda3/envs/deeplearning/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Load the instruction-tuning dataset
The dataset is provided as a JSONL file. We convert it to a *HuggingFace Dataset* for easier access and processing.

In [3]:
dataset = load_dataset("json", data_files={
    "train": r"instruction-tuning dataset/train.jsonl",
    "dev":   r"instruction-tuning dataset/dev.jsonl",
    "test":  r"instruction-tuning dataset/test.jsonl",
})

# Inspect dataset structure
print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]


Generating train split: 767 examples [00:00, 158693.33 examples/s]


Generating dev split: 0 examples [00:00, ? examples/s]


Generating dev split: 164 examples [00:00, 76378.62 examples/s]


Generating test split: 0 examples [00:00, ? examples/s]


Generating test split: 166 examples [00:00, 83413.74 examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 767
    })
    dev: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 164
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 166
    })
})


In [4]:
# Inspect a single training example
print(dataset["train"][0])

{'instruction': 'Explain the answer to the question below based strictly on the provided document.', 'input': 'Question:\nhas anyone investigated the effect of shock generated vorticity on heat\ntransfer to a blunt body .\n\nDocument:\nheat transfer at the forward stagnation point of blunt\nbodies .\nheat transfer at the forward stagnation point of blunt\nbodies .\n  relations are presented for the calculation of heat transfer at\nthe forward stagnation point of both two-dimensional and axially\nsymmetric blunt bodies .  the relations for the heat transfer, which were\nobtained from exact solutions to the equations of the laminar boundary\nlayer, are presented in terms of the local velocity gradient at the\nstagnation point .  these exact solutions include effects of variation\nof fluid properties, prandtl number, and transpiration cooling .\nexamples illustrating the calculation procedure are also included .', 'output': 'The provided document does not mention any investigation of **sh

## Task 0: Prompt Formatting

Before tokenization, each data sample is formatted as a single sequence by concatenating the instruction, input, and response with section headers.

**Note**
1. Section headers (e.g., `### Instruction:`, `### Input:`, `### Response:`) act as structural delimiters. The model learns to associate these markers with the roles of different segments and to generate the response following the ### Response: header.
2. The template must be identical at inference time


In [5]:
def format_prompt(sample: dict) -> tuple[str, str]:
    """
    Convert a raw dataset sample into a (prompt, response) string pair.

    Requirements:
        - Combine the 'instruction' and 'input' fields into a prompt string
            using the following fixed template
        - The prompt must end with the '### Response:\n' header and no response text.
        - Take the 'output' field directly as the response string y.
        - A sample complete template to use:
                ### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n{Response}
        - The same template must be used at inference time with exactly the section headersand and newlines and {response} being generated.

    Args:
        sample: A single dataset sample with keys 'instruction', 'input',
                 and 'output'.

    Returns:
        A tuple (prompt, response) where:                                                                                                                                                                                       
            prompt   : str  — the full instruction string x, ending with                                                                                                                                                                                                                                                                                                                                                                                                                                                        
                              the response header and no response text.
            response : str  — the raw output string y.

    Note:
        The prompt and response are returned separately as a tuple to identify the
        prompt boundary m, which is required for instruction-token masking (Task 1)
        and response-only perplexity (Task 3).
    """
    
    instruction = sample['instruction']
    input_text = sample['input']
    output = sample['output']
    
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    
    return prompt, output                                                                                                                                        

## Load the tokenzier

In [6]:
# Since the same tokenizer is reused across tasks (Task 1, 2, and 3), we load it once here at the top rather than repeating it in each task.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")

# Qwen tokenizer may not have a pad token — reuse eos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## Task 1: Completion-only tuning

Fine-tune using **only response tokens** $y$ by masking the instruction tokens in the labels:

$$\mathcal{L}_\theta = -\frac{1}{n} \sum_{i=1}^{n} \log P(y_i \mid x_{1:m},\, y_{<i})$$


### Task 1a: Tokenization/ Data preparation for instruction masking

PyTorch expects all inputs to be tensors of fixed length. We therefore tokenize each formatted sequence $s = (x, y)$ into `input_ids`, pad to `MAX_LENGTH`, and construct an `attention_mask` to ignore padding tokens and a `labels` tensor for the loss computation.

In [7]:
def tokenize_completion_only(sample: dict, tokenizer, max_length: int) -> dict[str, list[int]]:
    """
    Tokenize one dataset sample for Task 1 (completion-only tuning).

    The full token sequence s = [x; y] is constructed by concatenating
    the prompt tokens x (length m) and response tokens y (length n),
    then padded to max_length. The labels tensor mirrors input_ids but
    masks all instruction and padding positions with -100 so that the
    cross-entropy loss is computed only over response tokens y.

    Args:
        sample: A single dataset record with keys 'instruction', 'input',
                 and 'output'.
        tokenizer: The tokenizer to use for encoding the text.
        max_length: Maximum sequence length for truncation and padding.

    Returns:
        A dict with three equal-length lists (length is max_length):
            input_ids      – token ids for the full sequence [x; y],
                             right-padded with pad_token_id.
            attention_mask – 1 for real tokens, 0 for padding positions.
            labels         – token ids for response positions only;
                             -100 everywhere else (ignored by the loss).

    """
    prompt, response = format_prompt(sample)
    
    # Tokenize the full sequence with truncation and padding
    encoding = tokenizer(prompt + response, truncation=True, max_length=max_length, padding='max_length')
    input_ids = encoding.input_ids
    
    # Create attention mask (1 for real tokens, 0 for padding)
    attention_mask = [1 if token_id != tokenizer.pad_token_id else 0 for token_id in input_ids]
    
    # Find response start position by tokenizing the prompt alone
    response_start = len(tokenizer(prompt).input_ids)
    
    # Find the end of actual content (before padding starts)
    seq_len = sum(attention_mask)
    
    # Create labels: -100 for prompt and padding, actual input_ids for response tokens
    labels = [-100] * len(input_ids)
    for i in range(response_start, seq_len):
        labels[i] = input_ids[i]
    
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

### Data Preparation

In [8]:
# Inspect sequence lengths in the training set
lengths = []
for sample in dataset["train"]:
    prompt, response = format_prompt(sample)
    full_text = prompt + response
    token_len = len(tokenizer(full_text).input_ids)
    lengths.append(token_len)

print(f"Min length: {min(lengths)}")
print(f"Max length: {max(lengths)}")
print(f"Mean length: {sum(lengths) / len(lengths):.1f}")
print(f"95th percentile: {sorted(lengths)[int(len(lengths) * 0.95)]}")
print(f"99th percentile: {sorted(lengths)[int(len(lengths) * 0.99)]}")

Min length: 78
Max length: 737
Mean length: 410.0
95th percentile: 623
99th percentile: 679


In [9]:
# Hint: inspect sequence lengths in the training set. Must be accomodations of both prompt and response tokens.
MAX_LENGTH = 700

# Prepare a dataset for training with input ids, attention mask and labels
remove_cols = dataset["train"].column_names
tokenized_completion = DatasetDict({
    split: dataset[split].map(
        tokenize_completion_only,
        fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
        remove_columns=remove_cols,
    )
    for split in ("train", "dev")
})

# Change to tensors
tokenized_completion.set_format("torch")
print(tokenized_completion)


Map:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                   | 0/767 [00:00<?, ? examples/s]


Map:  12%|████████████████████████████████████████████████████                                                                                                                                                                                                                                                                                                                                                                                     | 94/767 [00:00<00:00, 930.62 examples/s]


Map:  25%|█████████████████████████████████████████████████████████████████████████████████████████████████████████                                                                                                                                                                                                                                                                                                                               | 190/767 [00:00<00:00, 943.97 examples/s]


Map:  43%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                                                                                                                                                                                   | 326/767 [00:00<00:00, 916.92 examples/s]


Map:  55%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                                                                                                                               | 420/767 [00:00<00:00, 921.87 examples/s]

...


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 767/767 [00:00<00:00, 779.76 examples/s]


Map:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                   | 0/164 [00:00<?, ? examples/s]


Map:  57%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                                                                                     | 94/164 [00:00<00:00, 923.05 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 164/164 [00:00<00:00, 747.84 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 767
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 164
    })
})


### Task 1b: Load the model and apply a LoRA adapter

Implement `load_model` below to load the base model and wrap it with a LoRA adapter ([PEFT docs](https://huggingface.co/docs/peft)) before training. The same `load_model` will be reused for Task 2

**Why LoRA?**

This dataset contains only ~767 training examples. Fine-tuning all 500 M+ parameters of Qwen1.5-0.5B on such a small set causes **overfitting**: the model memorises training sequences but generalises poorly, visible as a widening gap between `train_loss` and `dev_loss`.

[LoRA (Low-Rank Adaptation)](https://huggingface.co/docs/peft/main/en/conceptual_guides/lora) addresses this by **freezing** the original weights and injecting small trainable low-rank matrices into selected projection layers:

$$W' = W_0 + BA, \quad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k}, \quad r \ll \min(d,\, k)$$

Only the delta $\Delta W = BA$ is trained, reducing trainable parameters from ~500 M to ~3.8 M (&lt;1 %). This acts as an implicit regulariser that makes fine-tuning feasible on a small dataset.

**Key LoRA hyperparameters**

| Parameter | Description |
|-----------|-------------|
| `r` | Rank of the low-rank matrices — controls adapter capacity |
| `lora_alpha` | Scaling factor applied to the LoRA output ($\alpha / r$) |
| `target_modules` | Which projection layers receive LoRA adapters |
| `lora_dropout` | Dropout applied to LoRA activations for regularisation |

After training, call `model.merge_and_unload()` to fold the LoRA weights back into the base model and return a standard HuggingFace checkpoint with no PEFT overhead.

In [10]:
def load_model(model_name: str = "Qwen/Qwen1.5-0.5B") -> AutoModelForCausalLM:
    """
    Load a pre-trained causal language model and wrap it with a LoRA adapter.

    Refer to the Task 1b description above for guidance on which LoRA
    hyperparameters to use and which projection layers to target.
    See the PEFT docs linked there for the LoraConfig API.

    Args:
        model_name: HuggingFace model ID to load (default "Qwen/Qwen1.5-0.5B").

    Returns:
        A PEFT-wrapped AutoModelForCausalLM ready for training.
    """
    # Load the pre-trained model
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Define LoRA configuration
    lora_config = LoraConfig(
        r=8,                          # Rank of the low-rank matrices
        lora_alpha=16,                # Scaling factor applied to LoRA output
        target_modules=["q_proj", "v_proj"],  # Target projection layers for LoRA
        lora_dropout=0.1,             # Dropout applied to LoRA activations
    )
    
    # Wrap the model with PEFT using the defined LoRA configuration
    peft_model = get_peft_model(model, lora_config)
    
    return peft_model

### Task 1c: Define the PyTorch training loop

You are allowed to add any other hyperparameters you wish to implement with plain PyTorch. The same `train_model` function will be used for Task 2.

Examples of additional hyperparameters you may consider (**Not mandatory**):
- **`lr_scheduler`**: Learning rate schedule such as cosine annealing or linear decay (e.g., `torch.optim.lr_scheduler.CosineAnnealingLR`)
- **`max_grad_norm`**: Gradient clipping threshold to stabilize training (e.g., `torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm=1.0)`)
- **`weight_decay`**: L2 regularization strength passed to AdamW (e.g., `weight_decay=0.1`)
- **`accumulation_steps`**: Number of steps to accumulate gradients before updating, simulating a larger batch size (Effective batch size = batch_size × accumulation_steps. For example, `accumulation_steps=4` with `batch_size=4`, effective batch size = 16). This is useful for avoiding CUDA out-of-memory (OOM) errors when larger batches don’t fit in GPU memory.

In [11]:
# You can change the function definition to add more params if you wish so
def train_model(
    model: AutoModelForCausalLM,
    tokenized_dataset: dict,
    epochs: int = 3,
    batch_size: int = 4,
    lr: float = 2e-5,
    accumulation_steps: int = 4,
) -> AutoModelForCausalLM:
    """
    Fine-tune a causal language model using a plain PyTorch training loop.

    Args:
        model:             A pre-loaded AutoModelForCausalLM instance to fine-tune.
        tokenized_dataset: A DatasetDict with 'train' and 'dev' splits, each
                           containing tensors: input_ids, attention_mask, labels.
        epochs:            Number of full passes over the training set (default 3).
        batch_size:        Number of examples per gradient update step (default 8).
        lr:                Peak learning rate for AdamW (default 2e-5).

    Returns:
        The fine-tuned model (modified in-place; returned for convenience).

    
    Requirements:
        - The training loop MUST include both a training phase and a dev evaluation
          phase for every epoch.
        - After each epoch, you MUST print both train_loss and dev_loss.   


    Note:
        1. Initialize an AdamW optimizer with weight decay.
        2. Create DataLoaders for the train and dev splits.
        3. For each epoch:
            a. Training loop:
                - Forward pass to get raw logits.
                - Apply the causal shift: logit at position t predicts token t+1.
                - Compute cross-entropy loss, ignoring positions where labels == -100. 
                  (Padding and masked positions are ignored via ignore_index=-100 in F.cross_entropy.)
                - Backward pass and optimizer step.
            b. Evaluation loop (no gradients):
                - Compute dev loss using the same procedure.
            c. Print metrics after each epoch:
                  train_loss, dev_loss


    """
    # Move model to the appropriate device
    model.to(DEVICE)
    
    # Initialize an AdamW optimizer with weight decay
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Create DataLoaders for the train and dev splits
    train_loader = DataLoader(tokenized_dataset["train"], batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(tokenized_dataset["dev"], batch_size=batch_size)
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        num_train_batches = 0
        
        optimizer.zero_grad()  # Zero gradients at the start of each epoch
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training")):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            # Forward pass to get raw logits
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Apply the causal shift: logit at position t predicts token t+1
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            
            # Compute cross-entropy loss, ignoring positions where labels == -100
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100
            )
            
            # Scale loss by accumulation steps for correct gradient averaging
            loss = loss / accumulation_steps
            
            # Backward pass (accumulate gradients)
            loss.backward()
            
            total_train_loss += loss.item() * accumulation_steps  # Unscale for logging
            num_train_batches += 1
            
            # Optimizer step every accumulation_steps
            if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad()
        
        avg_train_loss = total_train_loss / num_train_batches
        
        # Evaluation loop
        model.eval()
        total_dev_loss = 0
        num_dev_batches = 0
        
        with torch.no_grad():
            for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1}/{epochs} - Evaluating"):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                
                # Forward pass to get raw logits
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                
                # Apply the causal shift
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels[:, 1:].contiguous()
                
                # Compute cross-entropy loss
                loss = F.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                    ignore_index=-100
                )
                
                total_dev_loss += loss.item()
                num_dev_batches += 1
        
        avg_dev_loss = total_dev_loss / num_dev_batches
        
        # Print metrics after each epoch
        print(f"Epoch {epoch+1}/{epochs} - train_loss: {avg_train_loss:.4f}, dev_loss: {avg_dev_loss:.4f}")
    
    return model

### Task 1d: Fine-tune and save the completion-only model

In [12]:
model_completion = load_model()
# Train with effective batch size of 16 (batch_size=4 × accumulation_steps=4)
model_completion = train_model(model_completion, tokenized_completion, epochs=3, batch_size=4, lr=2e-5, accumulation_steps=16)

# Merge LoRA weights into base model
model_completion = model_completion.merge_and_unload()

# Save the model
model_completion.save_pretrained("ckpt-completion")

del model_completion
torch.cuda.empty_cache()
print("\nTask 1 complete. Model saved to ckpt-completion/ and freed from memory.")


Loading weights:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                              | 0/291 [00:00<?, ?it/s]


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 291/291 [00:00<00:00, 9573.78it/s]


The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Epoch 1/3 - Training:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                         | 0/192 [00:00<?, ?it/s]


Epoch 1/3 - Training:   1%|██▏                                                                                                                                                                                                                                                                                                                                                                                                                            | 1/192 [00:27<1:26:22, 27.13s/it]

...

Epoch 3/3 - train_loss: 1.8341, dev_loss: 1.8548



Writing model shards:   0%|                                                                                                                                             | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.13s/it]


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.13s/it]


Task 1 complete. Model saved to ckpt-completion/ and freed from memory.


# Task 2: Full-prompt tuning

Fine-tune using **all tokens** (instruction + response):

$$\mathcal{L}_\theta = -\frac{1}{T} \sum_{t=1}^{T} \log P(s_t \mid s_{<t})$$

Here the labels are identical to `input_ids` (except padding which is masked with `-100`).  
The model must learn to predict instruction tokens as well as response tokens.

### Task 2a: Tokenize full sequence in labels

In [13]:
def tokenize_full_prompt(sample: dict, tokenizer, max_length: int) -> dict[str, list[int]]:
    """
    Tokenize one dataset sample for Task 2 (full-prompt tuning).

    Unlike completion-only tuning, the labels here cover the entire token
    sequence s = [x; y]. The model is trained to predict every token,
    including instruction tokens.


    Args:
        sample: A single dataset record with keys 'instruction', 'input',
                 and 'output'.
        tokenizer: The tokenizer to use for encoding the text.
        max_length: Maximum sequence length for truncation and padding.

    Returns:
        A dict with three equal-length lists (length is max_length):
            input_ids      – token ids for the full sequence [x; y],
                             right-padded with pad_token_id.
            attention_mask – 1 for real tokens, 0 for padding positions.
            labels         – same as input_ids for all real token positions;
                             -100 at padding positions (ignored by the loss).
    """
    prompt, response = format_prompt(sample)
    
    # Tokenize the full sequence with truncation and padding
    encoding = tokenizer(prompt + response, truncation=True, max_length=max_length, padding='max_length')
    input_ids = encoding.input_ids
    attention_mask = encoding.attention_mask
    
    # Create labels: same as input_ids for real tokens (mask=1), -100 for padding (mask=0)
    labels = [token_id if mask == 1 else -100 for token_id, mask in zip(input_ids, attention_mask)]
    
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

### Data Preparation

In [14]:
#TODO: Set MAX_LENGTH chosen before 
MAX_LENGTH = 700

# Prepare train/dev dataset
remove_cols = dataset["train"].column_names
tokenized_full = DatasetDict({
    split: dataset[split].map(
        tokenize_full_prompt,
        fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
        remove_columns=remove_cols,
    )
    for split in ("train", "dev")
})

# Convert to tensors
tokenized_full.set_format("torch")
print(tokenized_full)


Map:   0%|                                                                                                                                                     | 0/767 [00:00<?, ? examples/s]


Map:  29%|████████████████████████████████████████▏                                                                                                | 225/767 [00:00<00:00, 2221.45 examples/s]


Map:  72%|███████████████████████████████████████████████████████████████████████████████████████████████████▎                                     | 556/767 [00:00<00:00, 2205.11 examples/s]


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 767/767 [00:00<00:00, 1552.11 examples/s]


Map:   0%|                                                                                                                                                     | 0/164 [00:00<?, ? examples/s]


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 164/164 [00:00<00:00, 1544.30 examples/s]


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 164/164 [00:00<00:00, 1510.65 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 767
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 164
    })
})


### Task 2b: Fine-tune and save the full-prompt model

We reuse the same `load_model` and `train_model` function defined in Task 1.

In [15]:
model_full = load_model()

# TODO: Call train_model with the hyperparameters you chose
model_full = train_model(model_full, tokenized_full, epochs=3, batch_size=4, lr=2e-5, accumulation_steps=16)

# Merge LoRA weights into base model
model_full = model_full.merge_and_unload()

model_full.save_pretrained("ckpt-full")
del model_full
torch.cuda.empty_cache()
print("\nTask 2 complete. Model saved to ckpt-full/ and freed from memory.")


Loading weights:   0%|                                                                                                                                                | 0/291 [00:00<?, ?it/s]


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 291/291 [00:00<00:00, 7550.48it/s]


The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Epoch 1/3 - Training:   0%|                                                                                                                                           | 0/192 [00:00<?, ?it/s]


Epoch 1/3 - Training:   1%|▋                                                                                                                                | 1/192 [00:27<1:28:08, 27.69s/it]

...

Epoch 3/3 - train_loss: 2.7367, dev_loss: 2.7671



Writing model shards:   0%|                                                                                                                                             | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.16it/s]


Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.16it/s]


Task 2 complete. Model saved to ckpt-full/ and freed from memory.


## Task 3: Evaluation with Perplexity

We report two perplexity metrics on the **test set** for both fine-tuned models:

$$\text{PPL}_{\text{resp}} = \exp\!\left(\frac{1}{n}\sum_{t=m+1}^{m+n} -\log P(s_t \mid s_{<t})\right)$$

$$\text{PPL}_{\text{all}}  = \exp\!\left(\frac{1}{T}\sum_{t=1}^{T}   -\log P(s_t \mid s_{<t})\right)$$

- $\text{PPL}_{\text{resp}}$ — how well the model predicts **response tokens only**.
- $\text{PPL}_{\text{all}}$  — how well the model predicts the **entire sequence** (instruction + response).

### Task 3a: Tokenize the Test set

In [16]:
def tokenize_for_eval(sample: dict, tokenizer, max_length: int) -> dict[str, list[int] | int]:
    """
    Tokenize one test sample and record the instruction boundary for evaluation.

    Produces the same input_ids / attention_mask as tokenize_full_prompt, but
    also stores 'prompt_len' (m) so that compute_perplexities can separate
    instruction positions from response positions when computing PPL_resp.

    Args:
        sample: A single dataset record with keys 'instruction', 'input',
                 and 'output'.
        tokenizer: The tokenizer to use for encoding the text.
        max_length: Maximum sequence length for truncation and padding.

    Returns:
        A dict with:
            input_ids      – token ids for the full sequence [x; y],
                             right-padded to max_length.
            attention_mask – 1 for real tokens, 0 for padding positions.
            prompt_len     – m, the number of instruction tokens in input_ids
                             (before any response tokens begin).
    """
    prompt, response = format_prompt(sample)
    
    # Tokenize the full sequence with truncation and padding
    encoding = tokenizer(prompt + response, truncation=True, max_length=max_length, padding='max_length')
    input_ids = encoding.input_ids
    attention_mask = encoding.attention_mask
    
    # Calculate prompt length (number of tokens in the prompt)
    prompt_len = len(tokenizer(prompt).input_ids)
    
    return {"input_ids": input_ids, "attention_mask": attention_mask, "prompt_len": prompt_len}




### Test Data Preparation

In [17]:
#TODO: Set MAX_LENGTH chosen before 
MAX_LENGTH = 700

# Prepare test dataset
test_eval = dataset["test"].map(
    tokenize_for_eval,
    fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
    remove_columns=dataset["test"].column_names,
)
test_eval.set_format("torch")
print(f"Test examples: {len(test_eval)}")


Map:   0%|                                                                                                                                                     | 0/166 [00:00<?, ? examples/s]


Map:  80%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                           | 133/166 [00:00<00:00, 1318.11 examples/s]


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 166/166 [00:00<00:00, 1081.65 examples/s]

Test examples: 166


### Task 3b: Compute perplexity
We run a forward pass, compute per-token NLL with `F.cross_entropy(..., reduction='none')`, then accumulate separately over response positions and all positions.

In [18]:
@torch.no_grad()
def compute_perplexities(
    model: AutoModelForCausalLM,
    test_dataset: "datasets.Dataset",
    batch_size: int = 8,
) -> dict[str, float]:
    """
    Compute PPL_resp and PPL_all for a fine-tuned model on a test dataset.

    Both metrics are derived from per-token negative log-likelihoods (NLL)
    obtained via a single forward pass. The causal shift (logit at position t
    predicts token t+1) is applied manually so we have full control over which
    positions contribute to each metric.

    PPL_resp accumulates NLL only over response token positions t in (m, ..., T-1),
    implementing:
        PPL_resp = exp( 1/n * sum_{t=m+1}^{T} -log P(s_t | s_{<t}) )

    PPL_all accumulates NLL over all real (non-padding) token positions,
    implementing:
        PPL_all = exp( 1/T * sum_{t=1}^{T} -log P(s_t | s_{<t}) )

    NLLs are accumulated across the entire test set before taking the mean,
    so sequences of different lengths are weighted by token count rather than
    by sample count.

    Args:
        model:        A fine-tuned AutoModelForCausalLM to evaluate.
        test_dataset: A HuggingFace Dataset with columns:
                          input_ids      (LongTensor, shape T)
                          attention_mask (LongTensor, shape T)
                          prompt_len     (int) — number of instruction tokens m
        batch_size:   Number of samples processed per forward pass (default 8).

    Returns:
        A dict with two float entries:
            'ppl_resp' – perplexity over response tokens only.
            'ppl_all'  – perplexity over the full token sequence.

    Note:
        Be mindful of the causal shift when mapping prompt_len m to indices in
        the shifted sequence — shifted index i predicts original token i+1.
    """
    model.to(DEVICE)
    model.eval()
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    total_nll_resp = 0.0
    total_nll_all = 0.0
    total_tokens_resp = 0
    total_tokens_all = 0
    
    for batch in tqdm(test_loader, desc="Computing perplexities"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        prompt_lens = batch["prompt_len"]  # Shape: (batch_size,)
        
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Apply causal shift: logit at position t predicts token t+1
        shift_logits = logits[:, :-1, :].contiguous()  # (B, T-1, V)
        shift_labels = input_ids[:, 1:].contiguous()    # (B, T-1)
        shift_mask = attention_mask[:, 1:].contiguous()  # (B, T-1) - mask for target tokens
        
        # Compute per-token NLL with reduction='none'
        nll = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction='none'
        ).view(shift_logits.size(0), shift_logits.size(1))  # (B, T-1)
        
        # Accumulate NLL for each sample in batch
        for i in range(input_ids.size(0)):
            m = prompt_lens[i].item()  # prompt length for this sample
            mask_i = shift_mask[i].bool()  # (T-1,) - True for real tokens
            nll_i = nll[i]  # (T-1,)
            
            # PPL_all: accumulate NLL over all real token positions
            total_nll_all += nll_i[mask_i].sum().item()
            total_tokens_all += mask_i.sum().item()
            
            # PPL_resp: accumulate NLL over response token positions only
            # Response tokens are at original positions m, m+1, ..., T-1
            # In shifted space (shifted index i predicts token i+1):
            #   to predict token at position m, use shifted index m-1
            # So response indices in shifted space are >= m-1
            seq_len = shift_logits.size(1)  # T-1
            resp_indices = torch.arange(seq_len, device=DEVICE) >= (m - 1)
            resp_mask = resp_indices & mask_i
            total_nll_resp += nll_i[resp_mask].sum().item()
            total_tokens_resp += resp_mask.sum().item()
    
    # Compute perplexities: exp(mean NLL)
    ppl_resp = math.exp(total_nll_resp / total_tokens_resp) if total_tokens_resp > 0 else float('inf')
    ppl_all = math.exp(total_nll_all / total_tokens_all) if total_tokens_all > 0 else float('inf')
    
    return {'ppl_resp': ppl_resp, 'ppl_all': ppl_all}

### Task 3c: Run evaluation on both fine-tuned models

In [19]:
print("Evaluating completion-only model (Task 1)...")
# Load the saved model
model_completion = AutoModelForCausalLM.from_pretrained("ckpt-completion")
results_completion = compute_perplexities(model_completion, test_eval)
del model_completion
torch.cuda.empty_cache()

print("\nEvaluating full-prompt model (Task 2)...")
# Load the saved model
model_full = AutoModelForCausalLM.from_pretrained("ckpt-full")
results_full = compute_perplexities(model_full, test_eval)
del model_full
torch.cuda.empty_cache()

print(f"\nCompletion-only: ppl_resp={results_completion['ppl_resp']:.2f}, ppl_all={results_completion['ppl_all']:.2f}")
print(f"Full-prompt:     ppl_resp={results_full['ppl_resp']:.2f}, ppl_all={results_full['ppl_all']:.2f}")

Evaluating completion-only model (Task 1)...



Loading weights:   0%|                                                                                                                                                | 0/291 [00:00<?, ?it/s]


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 291/291 [00:00<00:00, 14025.35it/s]


The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Computing perplexities:   0%|                                                                                                                                          | 0/21 [00:00<?, ?it/s]

...


Computing perplexities:  90%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋            | 19/21 [03:19<00:20, 10.50s/it]


Computing perplexities:  95%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊      | 20/21 [03:29<00:10, 10.50s/it]


Computing perplexities: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [03:37<00:00,  9.76s/it]


Computing perplexities: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [03:37<00:00, 10.38s/it]


Completion-only: ppl_resp=6.26, ppl_all=16.19
Full-prompt:     ppl_resp=6.57, ppl_all=15.39


## Interpretation of Results

Fill in your numbers after running the evaluation cell and interpret your results:

| Model | $\text{PPL}_{\text{resp}}$ | $\text{PPL}_{\text{all}}$ |
|-------|:--------------------------:|:-------------------------:|
| Qwen1.5-0.5B-completion-only | 6.26 | 16.19 |
| Qwen1.5-0.5B-full-prompt     | 6.57 | 15.39 |

### Discussion

**Response-only perplexity ($\text{PPL}_{\text{resp}}$):**
The completion-only model achieves a lower $\text{PPL}_{\text{resp}}$ (6.26) compared to the full-prompt model (6.57). This is expected because the completion-only model focuses its entire training signal exclusively on predicting response tokens — by masking the instruction tokens from the loss, all gradient updates are directed toward improving response generation. The full-prompt model, by contrast, must split its limited LoRA capacity across learning both instruction patterns and response patterns, slightly diluting its response prediction ability.

**Full-sequence perplexity ($\text{PPL}_{\text{all}}$):**
The full-prompt model achieves a lower $\text{PPL}_{\text{all}}$ (15.39) compared to the completion-only model (16.19). This result is also expected: the full-prompt model is explicitly trained to predict all tokens including instruction tokens, so it naturally models the entire sequence better. The completion-only model never receives gradients from instruction positions, so it relies solely on the pre-trained weights for those tokens, leading to higher overall perplexity.

**Trade-off between the two strategies:**
The results illustrate a clear trade-off. Completion-only tuning is more **task-efficient** — it specializes the model for the downstream use case (generating responses) and achieves the best response quality as measured by $\text{PPL}_{\text{resp}}$. Full-prompt tuning trades a small amount of response quality for better **sequence-level modeling**, which can help the model better understand and condition on instruction structure. In practice, completion-only tuning is generally preferred for instruction-following tasks because at inference time we only care about the quality of generated responses, not the model's ability to predict the prompt itself.